In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn
import pennylane as qml
import optax
import numpy as np
import os
import glob

In [ ]:
# ==========================================
# 1. PATCH DATA LOADER (The Key Difference)
# ==========================================
class PatchLoader:
    def __init__(self, root_dir, batch_size=32, patch_size=3):
        self.files = glob.glob(os.path.join(root_dir, "**/*.npz"), recursive=True)
        self.batch_size = batch_size
        self.patch_size = patch_size
        self.half_size = patch_size // 2
        
        # Load one file into memory to sample from (simplified for tutorial)
        # In a real app, you'd rotate through files
        if len(self.files) > 0:
            self.load_new_scene(0)
            
    def load_new_scene(self, idx):
        data = np.load(self.files[idx])
        img = data['image'] # (Channels, H, W) usually
        mask = data['label']
        
        # Handle dimensions (Sen2Fire can be (13, 512, 512))
        if img.shape[0] == 13:
            # SWIR Composite: B12, B8, B4 -> Indices 11, 7, 3
            img = img[[11, 7, 3], :, :] 
            img = np.transpose(img, (1, 2, 0)) # (H, W, 3)
        
        # Normalize
        img = img.astype(np.float32) / 10000.0
        self.img = np.clip(img, 0, 1)
        self.mask = mask.astype(np.float32)
        
        # Pre-calculate indices of Fire and Non-Fire pixels for balancing
        # We ignore edges to avoid boundary checks during simple sampling
        valid_mask = self.mask[self.half_size:-self.half_size, self.half_size:-self.half_size]
        
        # Get coordinates (shifted by half_size)
        self.fire_indices = np.argwhere(valid_mask == 1) + self.half_size
        self.bg_indices = np.argwhere(valid_mask == 0) + self.half_size

    def get_batch(self):
        # We want a BALANCED batch (50% Fire, 50% Background)
        # This solves the "97% background" problem of Sen2Fire
        n_fire = self.batch_size // 2
        n_bg = self.batch_size - n_fire
        
        # Randomly sample centers
        idx_fire = np.random.choice(len(self.fire_indices), n_fire)
        idx_bg = np.random.choice(len(self.bg_indices), n_bg)
        
        centers_fire = self.fire_indices[idx_fire]
        centers_bg = self.bg_indices[idx_bg]
        centers = np.vstack([centers_fire, centers_bg])
        
        patches = []
        labels = []
        
        for r, c in centers:
            # Extract 3x3 patch
            r_start, r_end = r - self.half_size, r + self.half_size + 1
            c_start, c_end = c - self.half_size, c + self.half_size + 1
            
            patch = self.img[r_start:r_end, c_start:c_end, :] # (3, 3, 3)
            patches.append(patch)
            labels.append(self.mask[r, c]) # Label is the center pixel
            
        return jnp.array(patches), jnp.array(labels)

In [ ]:
# ==========================================
# 2. THE QUANTUM PATCH CLASSIFIER
# ==========================================
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="jax")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

class QuantumPatchModel(nn.Module):
    @nn.compact
    def __call__(self, x):
        # Input x shape: (Batch, 3, 3, 3) -> Flatten to (Batch, 27)
        x_flat = x.reshape((x.shape[0], -1))
        
        # Classical Compression: 27 inputs -> 4 Qubits
        # This part runs on CPU
        x_embed = nn.Dense(n_qubits)(x_flat)
        x_embed = jnp.pi * nn.tanh(x_embed) # Scale to [-pi, pi] for rotation
        
        # Quantum Layer
        weights = self.param('weights', nn.initializers.uniform(scale=0.1), (2, n_qubits))
        
        # Vectorize over the batch
        q_out = jax.vmap(quantum_circuit, in_axes=(0, None))(x_embed, weights)
        q_out = jnp.stack(q_out, axis=-1) # (Batch, 4)
        
        # Classical Classification Head
        x = nn.Dense(1)(q_out)
        return x # Logits

In [ ]:
# ==========================================
# 3. TRAINING LOOP
# ==========================================
def train_patch_model():
    DATA_PATH = "./Sen2Fire" # <--- Update this
    
    # 1. Init Loader
    # If no data exists, this will crash. Ensure folder is correct.
    try:
        loader = PatchLoader(DATA_PATH, batch_size=32)
    except:
        print("Data not found. Please set DATA_PATH.")
        return

    # 2. Init Model
    model = QuantumPatchModel()
    dummy_x = jnp.zeros((1, 3, 3, 3))
    key = jax.random.PRNGKey(0)
    params = model.init(key, dummy_x)['params']
    
    tx = optax.adam(learning_rate=0.01)
    opt_state = tx.init(params)
    
    @jax.jit
    def train_step(params, opt_state, x, y):
        def loss_fn(p, x, y):
            logits = model.apply({'params': p}, x).squeeze()
            return jnp.mean(optax.sigmoid_binary_cross_entropy(logits, y))
        
        loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
        updates, new_opt_state = tx.update(grads, opt_state)
        return optax.apply_updates(params, updates), new_opt_state, loss

    print("Starting Balanced Patch Training...")
    for i in range(200): # 200 steps of 32 patches
        x_batch, y_batch = loader.get_batch()
        params, opt_state, loss = train_step(params, opt_state, x_batch, y_batch)
        
        if i % 20 == 0:
            print(f"Step {i} | Loss: {loss:.4f}")

In [ ]:
train_patch_model()